Cell 1 — Setup and load everything

In [ ]:
import torch.nn.functional as F
from torch import optim
from google.colab import drive
drive.mount('/content/drive')

import os, json, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

DRIVE_FOLDER = '/content/drive/MyDrive/traffic_project'

# Load config
with open(f'{DRIVE_FOLDER}/config.json') as f:
    config = json.load(f)

# Load 3D data array
print("Loading data_3d.npy...")
data_3d = np.load(f'{DRIVE_FOLDER}/data_3d.npy')
print(f"✅ data_3d shape: {data_3d.shape}")

# Load adjacency matrix
adj_tensor = torch.load(f'{DRIVE_FOLDER}/adj_tensor.pt')
print(f"✅ adj_tensor shape: {adj_tensor.shape}")

# Load scaler
with open(f'{DRIVE_FOLDER}/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Unpack config
N_SENSORS   = config['n_sensors']
N_TIMES     = config['n_timesteps']
N_FEATURES  = config['n_features']
INPUT_STEPS = config['input_steps']
PRED_STEPS  = config['pred_steps']
BATCH_SIZE  = config['batch_size']
TRAIN_END   = config['train_end_t']
VAL_END     = config['val_end_t']
SPEED_IDX   = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✅ Device: {device}")
print(f"   Sensors={N_SENSORS}, Times={N_TIMES}, Features={N_FEATURES}")

# Pre-compute normalized adjacency (always needed)
def normalize_adjacency(adj):
    A     = adj + torch.eye(adj.size(0), device=adj.device)
    deg   = A.sum(dim=1)
    D_inv = torch.diag(deg.pow(-0.5))
    return D_inv @ A @ D_inv

adj_norm = normalize_adjacency(adj_tensor).to(device)
print(f"✅ adj_norm ready: {adj_norm.shape}  device: {adj_norm.device}")

Mounted at /content/drive
Loading data_3d.npy...
✅ data_3d shape: (52116, 325, 12)
✅ adj_tensor shape: torch.Size([325, 325])

✅ Device: cuda
   Sensors=325, Times=52116, Features=12
✅ adj_norm ready: torch.Size([325, 325])  device: cuda:0


## Cell 2 — Dataset & DataLoaders
Defines `TrafficDataset` (sliding-window wrapper) and builds train/val/test `DataLoader` instances.

> `max_samples` limits dataset size for faster ablation experiments — set to `None` for full training.

In [ ]:
# ── Dataset class ──────────────────────────────────────────────────────────
# TrafficDataset wraps the 3D numpy array (T, N, F) and returns sliding windows:
#   x: (input_steps=12, N_sensors=325, N_features=12)  ← model input
#   y: (pred_steps=12, N_sensors=325)                  ← speed targets only
# max_samples: optional cap for faster ablation/debugging runs
# ─────────────────────────────────────────────────────────────────────────────
class TrafficDataset(Dataset):
    def __init__(self, data_3d, indices, input_steps=12, pred_steps=12,
                 max_samples=None):
        self.data        = data_3d
        self.input_steps = input_steps
        self.pred_steps  = pred_steps

        # Subsample indices if max_samples is set
        if max_samples and len(indices) > max_samples:
            idx = np.random.choice(len(indices), max_samples, replace=False)
            self.indices = [indices[i] for i in idx]
        else:
            self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        t = self.indices[i]
        x = self.data[t - self.input_steps : t]
        y = self.data[t : t + self.pred_steps, :, SPEED_IDX]
        return (torch.tensor(x, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32))

train_indices = list(range(INPUT_STEPS, TRAIN_END - PRED_STEPS))
val_indices   = list(range(TRAIN_END + INPUT_STEPS, VAL_END - PRED_STEPS))
test_indices  = list(range(VAL_END + INPUT_STEPS, N_TIMES - PRED_STEPS))

# Use 3000 train samples and 1000 val samples per epoch
# This is statistically representative of the full dataset
# and keeps each epoch to ~3-5 minutes
TRAIN_SAMPLES_PER_EPOCH  = 12000
VAL_SAMPLES_PER_EPOCH    = 3000

train_dataset = TrafficDataset(data_3d, train_indices,
                               max_samples=TRAIN_SAMPLES_PER_EPOCH)
val_dataset   = TrafficDataset(data_3d, val_indices,
                               max_samples=VAL_SAMPLES_PER_EPOCH)
test_dataset  = TrafficDataset(data_3d, test_indices)  # full test set

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ DataLoaders ready (subsampled)")
print(f"   Train per epoch: {len(train_dataset):,} samples "
      f"({len(train_loader)} batches)")
print(f"   Val per epoch:   {len(val_dataset):,} samples "
      f"({len(val_loader)} batches)")
print(f"   Test (full):     {len(test_dataset):,} samples")

✅ DataLoaders ready (subsampled)
   Train per epoch: 12,000 samples (375 batches)
   Val per epoch:   3,000 samples (94 batches)
   Test (full):     10,400 samples


Cell 4 — The Temporal Transformer

In [ ]:
# ── TRANSFORMER: Temporal Self-Attention ─────────────────────
#
# Plain English: the Transformer reads the 12-step history for
# each sensor and decides — via "attention" — which past moments
# matter most for predicting the future.
#
# Example: if there was a slowdown 3 steps ago, the model learns
# to pay more attention to that moment when making predictions.
#
# Multi-head attention: runs several attention computations in
# parallel, each focusing on different aspects (speed trend,
# weather change, time-of-day pattern, etc.)

class TemporalTransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, dropout=0.1):
        super().__init__()
        """
        d_model:  feature dimension (must be divisible by n_heads)
        n_heads:  number of parallel attention heads
        n_layers: how many Transformer encoder layers to stack
        """
        # Positional encoding: tells the model the ORDER of timesteps
        # (Transformers don't inherently know that step 3 comes after step 2)
        self.pos_embedding = nn.Parameter(
            torch.randn(1, 12, d_model) * 0.1   # learnable, shape (1, T, d_model)
        )

        # Stack of Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,   # standard: 4× the model dim
            dropout=dropout,
            batch_first=True               # input: (batch, seq, features)
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        x: (batch * n_sensors, input_steps, d_model)
        Returns: (batch * n_sensors, d_model)
              — one vector per sensor summarizing the full history
        """
        # Add positional encoding
        x = x + self.pos_embedding[:, :x.size(1), :]

        # Run through Transformer
        out = self.transformer(x)   # (B*N, T, d_model)

        # Take the last timestep as the summary vector
        # (like reading a book and summarizing the last page)
        return self.norm(out[:, -1, :])   # (B*N, d_model)


print("✅ Temporal Transformer defined")

✅ Temporal Transformer defined


Cell 5 — The Full Model

In [ ]:
import torch.nn.functional as F

class DiffusionConvLayer(nn.Module):
    """
    Diffusion convolution: runs graph conv in BOTH directions
    (forward = traffic flow direction, backward = reverse)
    Then combines both with a learnable weight.
    This is what makes DCRNN powerful — it captures how
    congestion propagates both upstream and downstream.
    """
    def __init__(self, d_model, n_hops=2):
        super().__init__()
        self.n_hops = n_hops
        # Separate weights for forward and backward propagation
        self.fwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.bwd_linears = nn.ModuleList([
            nn.Linear(d_model, d_model, bias=False) for _ in range(n_hops)
        ])
        self.out = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, adj_fwd, adj_bwd):
        """
        x:       (B*T, N, d_model)
        adj_fwd: (N, N) — forward adjacency (row-normalized)
        adj_bwd: (N, N) — backward adjacency (transpose, row-normalized)
        """
        # Forward diffusion: congestion spreading downstream
        h_fwd = x
        for layer in self.fwd_linears:
            h_fwd = F.relu(layer(
                torch.einsum('nm, bmd -> bnd', adj_fwd, h_fwd)
            ))

        # Backward diffusion: upstream influence
        h_bwd = x
        for layer in self.bwd_linears:
            h_bwd = F.relu(layer(
                torch.einsum('nm, bmd -> bnd', adj_bwd, h_bwd)
            ))

        # Combine forward and backward
        combined = torch.cat([h_fwd, h_bwd], dim=-1)  # (B*T, N, 2*d)
        out = self.out(combined)                        # (B*T, N, d)
        return self.norm(out + x)                       # residual connection


class SpatioTemporalTransformer(nn.Module):
    def __init__(
        self,
        n_features,
        d_model,
        n_heads,
        n_gnn_layers,
        n_tf_layers,
        n_sensors,
        pred_steps,
        dropout=0.1
    ):
        super().__init__()
        self.n_sensors  = n_sensors
        self.pred_steps = pred_steps
        self.d_model    = d_model

        # 1. Input projection
        self.input_proj = nn.Linear(n_features, d_model)

        # 2. Diffusion GNN layers
        self.gnn_layers = nn.ModuleList([
            DiffusionConvLayer(d_model, n_hops=2)
            for _ in range(n_gnn_layers)
        ])

        # Gated fusion: learns HOW MUCH spatial context to use
        # If gate=1 → use full spatial output
        # If gate=0 → use original features (bypass spatial)
        self.spatial_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.Sigmoid()
        )

        # 3. Temporal Transformer
        self.pos_embedding = nn.Parameter(
            torch.randn(1, INPUT_STEPS, d_model) * 0.02
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_tf_layers
        )
        self.tf_norm = nn.LayerNorm(d_model)

        # 4. Prediction head
        self.pred_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps)
        )

    def forward(self, x, adj_fwd, adj_bwd):
        """
        x:       (B, T, N, n_feat)
        adj_fwd: (N, N) — forward normalized adjacency
        adj_bwd: (N, N) — backward normalized adjacency
        """
        B, T, N, n_feat = x.shape

        # ── Step 1: Input projection ──────────────────────────
        x = self.input_proj(x)              # (B, T, N, d_model)
        x_orig = x.clone()                  # save for gated fusion

        # ── Step 2: Diffusion GNN ─────────────────────────────
        x_flat = x.reshape(B * T, N, self.d_model)

        for gnn in self.gnn_layers:
            x_flat = gnn(x_flat, adj_fwd, adj_bwd)

        x_spatial = x_flat.reshape(B, T, N, self.d_model)

        # Gated fusion: blend spatial output with original features
        gate_input = torch.cat([x_spatial, x_orig], dim=-1)  # (B,T,N,2d)
        gate       = self.spatial_gate(gate_input)             # (B,T,N,d)
        x          = gate * x_spatial + (1 - gate) * x_orig   # (B,T,N,d)

        # ── Step 3: Temporal Transformer ─────────────────────
        x = x.permute(0, 2, 1, 3).reshape(B * N, T, self.d_model)
        x = x + self.pos_embedding[:, :T, :]
        x = self.transformer(x)
        x = self.tf_norm(x[:, -1, :])

        # ── Step 4: Prediction head ───────────────────────────
        pred = self.pred_head(x)
        pred = pred.reshape(B, N, self.pred_steps)
        pred = pred.permute(0, 2, 1)
        return pred


# Pre-compute BOTH forward and backward adjacency matrices
def normalize_adjacency_directed(adj):
    """Row-normalize: each row sums to 1"""
    row_sum = adj.sum(dim=1, keepdim=True).clamp(min=1e-8)
    return adj / row_sum

adj_fwd = normalize_adjacency_directed(adj_tensor).to(device)
adj_bwd = normalize_adjacency_directed(adj_tensor.T).to(device)

print(f"✅ Forward adjacency: {adj_fwd.shape}")
print(f"✅ Backward adjacency: {adj_bwd.shape}")

# Instantiate model
model = SpatioTemporalTransformer(
    n_features   = N_FEATURES,
    d_model      = 64,
    n_heads      = 4,
    n_gnn_layers = 2,
    n_tf_layers  = 2,
    n_sensors    = N_SENSORS,
    pred_steps   = PRED_STEPS,
    dropout      = 0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Improved SpatioTemporalTransformer ready")
print(f"   Parameters: {total_params:,}")

✅ Forward adjacency: torch.Size([325, 325])
✅ Backward adjacency: torch.Size([325, 325])
✅ Improved SpatioTemporalTransformer ready
   Parameters: 161,964


Cell 6 — Quick forward pass test (sanity check before training)

In [ ]:
model.eval()
with torch.no_grad():
    x_test = torch.randn(4, INPUT_STEPS, N_SENSORS, N_FEATURES).to(device)
    out    = model(x_test, adj_fwd, adj_bwd)

print(f"✅ Forward pass test passed")
print(f"   Input:  {x_test.shape}")
print(f"   Output: {out.shape}   — expected (4, 12, 325)")
assert out.shape == (4, PRED_STEPS, N_SENSORS)

✅ Forward pass test passed
   Input:  torch.Size([4, 12, 325, 12])
   Output: torch.Size([4, 12, 325])   — expected (4, 12, 325)


Cell 7 — Training loop with checkpointing

In [ ]:
N_EPOCHS   = 150
LR         = 3e-4
CHECKPOINT = f'{DRIVE_FOLDER}/stgt_best.pt'

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4
)
loss_fn        = nn.HuberLoss(delta=1.0)
best_val_loss  = float('inf')
train_losses   = []
val_losses     = []
patience_count = 0
EARLY_STOP     = 15

print(f"Training SpatioTemporalTransformer")
print(f"  Epochs:              {N_EPOCHS}")
print(f"  Samples/epoch train: {TRAIN_SAMPLES_PER_EPOCH:,}")
print(f"  Samples/epoch val:   {VAL_SAMPLES_PER_EPOCH:,}")
print(f"  Device:              {device}")
print(f"\n{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'LR':>12}")
print("-"*48)

import time

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()

    # ── Re-sample training data each epoch ────────────────
    # This ensures the model sees different timesteps every epoch
    train_dataset = TrafficDataset(data_3d, train_indices,
                                   max_samples=TRAIN_SAMPLES_PER_EPOCH)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                               shuffle=True, num_workers=2, pin_memory=True)

    # ── Train ──────────────────────────────────────────────
    model.train()
    epoch_train = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        pred = model(x_batch, adj_fwd, adj_bwd)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_train += loss.item()

    avg_train = epoch_train / len(train_loader)

    # ── Validate ───────────────────────────────────────────
    model.eval()
    epoch_val = 0

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            pred = model(x_batch, adj_fwd, adj_bwd)
            loss    = loss_fn(pred, y_batch)
            epoch_val += loss.item()

    avg_val = epoch_val / len(val_loader)
    elapsed = time.time() - t0

    train_losses.append(avg_train)
    val_losses.append(avg_val)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"{epoch:>6}   {avg_train:>10.4f}   {avg_val:>10.4f}   "
          f"{current_lr:>10.6f}   ({elapsed:.0f}s)")

    scheduler.step(avg_val)

    # Save best model
    if avg_val < best_val_loss:
        best_val_loss  = avg_val
        patience_count = 0
        torch.save({
            'epoch':             epoch,
            'model_state_dict':  model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss':          best_val_loss,
        }, CHECKPOINT)
        print(f"         ✅ Best model saved (val={best_val_loss:.4f})")
    else:
        patience_count += 1
        if patience_count >= EARLY_STOP:
            print(f"\n⏹ Early stopping (no improvement for {EARLY_STOP} epochs)")
            break

print(f"\n✅ Training complete. Best val loss: {best_val_loss:.4f}")
np.save(f'{DRIVE_FOLDER}/stgt_train_losses.npy', np.array(train_losses))
np.save(f'{DRIVE_FOLDER}/stgt_val_losses.npy',   np.array(val_losses))

Training SpatioTemporalTransformer
  Epochs:              150
  Samples/epoch train: 12,000
  Samples/epoch val:   3,000
  Device:              cuda

 Epoch   Train Loss     Val Loss           LR
------------------------------------------------
     1       0.1396       0.1044     0.000300   (90s)
         ✅ Best model saved (val=0.1044)
     2       0.0897       0.0987     0.000300   (89s)
         ✅ Best model saved (val=0.0987)
     3       0.0852       0.0961     0.000300   (89s)
         ✅ Best model saved (val=0.0961)
     4       0.0844       0.0946     0.000300   (89s)
         ✅ Best model saved (val=0.0946)
     5       0.0827       0.0939     0.000300   (89s)
         ✅ Best model saved (val=0.0939)
     6       0.0818       0.0941     0.000300   (89s)
     7       0.0791       0.0921     0.000300   (89s)
         ✅ Best model saved (val=0.0921)
     8       0.0797       0.0939     0.000300   (89s)
     9       0.0778       0.0890     0.000300   (89s)
         ✅ Best model s

Cell 8 — Evaluate on test set

In [ ]:
# Load best checkpoint
checkpoint = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"✅ Loaded best model from epoch {checkpoint['epoch']}")
print(f"   Best val loss: {checkpoint['val_loss']:.4f}")
print(f"\nEvaluating on test set...")

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mape(y_true, y_pred, eps=1e-5):
    mask = np.abs(y_true) > eps
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

all_preds, all_trues = [], []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        pred = model(x_batch, adj_fwd, adj_bwd)
        all_preds.append(pred.cpu().numpy())
        all_trues.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)  # (n_test, 12, 325)
all_trues = np.concatenate(all_trues, axis=0)

# Results at each horizon
print(f"\n{'Model':<30} {'MAE':>8} {'RMSE':>8} {'MAPE':>8}")
print("-"*58)

stgt_results = {}
for h, label in [(2, '15 min'), (5, '30 min'), (11, '60 min')]:
    y_t = all_trues[:, h, :].flatten()
    y_p = all_preds[:, h, :].flatten()
    m, r, p = mae(y_t, y_p), rmse(y_t, y_p), mape(y_t, y_p)
    print(f"  STGTransformer @ {label:<10}  {m:>8.4f}  {r:>8.4f}  {p:>7.2f}%")
    stgt_results[label] = {'MAE': float(m), 'RMSE': float(r), 'MAPE': float(p)}

# Save predictions
np.save(f'{DRIVE_FOLDER}/stgt_test_preds.npy', all_preds)
np.save(f'{DRIVE_FOLDER}/stgt_test_trues.npy', all_trues)
print(f"\n✅ Predictions saved to Drive")

NameError: name 'CHECKPOINT' is not defined

Cell 9 — Full comparison table

In [ ]:
# Load baseline results
with open(f'{DRIVE_FOLDER}/baseline_results.json') as f:
    baseline = json.load(f)

print("\n" + "="*65)
print("FULL MODEL COMPARISON (MAE — lower is better)")
print("="*65)
print(f"{'Model':<28} {'15 min':>10} {'30 min':>10} {'60 min':>10}")
print("-"*65)

rows = {
    'Historical Average':    baseline['HA'],
    'ARIMA':                 baseline['ARIMA'],
    'LSTM (no spatial)':     baseline['LSTM'],
    'STGTransformer (ours)': {k: v['MAE'] for k, v in stgt_results.items()},
}

for name, res in rows.items():
    vals = [res.get(h, res.get(h, 0)) for h in ['15 min', '30 min', '60 min']]
    marker = ' ← OURS' if 'STG' in name else ''
    print(f"  {name:<26} {vals[0]:>10.4f} {vals[1]:>10.4f} {vals[2]:>10.4f}{marker}")

print("="*65)

# Improvement over LSTM
lstm_15 = baseline['LSTM']['15 min']
stgt_15 = stgt_results['15 min']['MAE']
improvement = (lstm_15 - stgt_15) / lstm_15 * 100
print(f"\n📈 Improvement over LSTM at 15 min: {improvement:.1f}%")

# Save all results
all_results = {
    'HA':           baseline['HA'],
    'ARIMA':        baseline['ARIMA'],
    'LSTM':         baseline['LSTM'],
    'STGTransformer': {k: v['MAE'] for k, v in stgt_results.items()},
}
with open(f'{DRIVE_FOLDER}/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n✅ All results saved to all_results.json")
print(f"✅ Phase 4 complete — ready for Phase 5 (Multi-modal ablation + SHAP)")

NameError: name 'DRIVE_FOLDER' is not defined